## CBoW model ko train karna

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) ka hissa hai.

Is example mein hum CBoW language model ko train karne par focus karenge, taaki hum apna Word2Vec embedding space bana sakein. Hum AG News dataset ko text ke source ke roop mein use karenge.

In [ ]:
import torch
import torchtext
import os
import collections
import builtins
import random
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Pehle chaliye apna dataset load karte hain aur tokenizer aur vocabulary define karte hain. Hum `vocab_size` ko 5000 par set karenge taaki calculations thodi limited rahein.

In [ ]:
def load_dataset(ngrams = 1, min_freq = 1, vocab_size = 5000 , lines_cnt = 500):
    tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
    print("Loading dataset...")
    test_dataset, train_dataset  = torchtext.datasets.AG_NEWS(root='./data')
    train_dataset = list(train_dataset)
    test_dataset = list(test_dataset)
    classes = ['World', 'Sports', 'Business', 'Sci/Tech']
    print('Building vocab...')
    counter = collections.Counter()
    for i, (_, line) in enumerate(train_dataset):
        counter.update(torchtext.data.utils.ngrams_iterator(tokenizer(line),ngrams=ngrams))
        if i == lines_cnt:
            break
    vocab = torchtext.vocab.Vocab(collections.Counter(dict(counter.most_common(vocab_size))), min_freq=min_freq)
    return train_dataset, test_dataset, classes, vocab, tokenizer

In [ ]:
train_dataset, test_dataset, _, vocab, tokenizer = load_dataset()

In [ ]:
def encode(x, vocabulary, tokenizer = tokenizer):
    return [vocabulary[s] for s in tokenizer(x)]

## CBoW model

CBoW $2N$ neighboring words ke basis par ek word ko predict karna seekhta hai. Jaise, jab $N=1$ ho, toh humein sentence *I like to train networks* se yeh pairs milenge: (like,I), (I, like), (to, like), (like,to), (train,to), (to, train), (networks, train), (train,networks). Yahan pehla word neighboring word hai jo input ke roop mein use hua hai, aur doosra word woh hai jise hum predict kar rahe hain.

Agla word predict karne ke liye network banane ke liye, humein neighboring word ko input dena hoga aur word number ko output ke roop mein lena hoga. CBoW network ka structure yeh hai:

* Input word ko embedding layer se guzaara jaata hai. Yahi embedding layer hamara Word2Vec embedding hoga, isliye hum ise alag se `embedder` variable ke roop mein define karenge. Is example mein hum embedding size = 30 use karenge, halaanki aap higher dimensions ke saath experiment karna chahenge (real Word2Vec mein 300 hota hai).
* Embedding vector ko phir ek linear layer mein pass kiya jayega, jo output word ko predict karegi. Isliye ismein `vocab_size` neurons honge.

Output ke liye, agar hum `CrossEntropyLoss` ko loss function ke roop mein use karte hain, toh humein expected result ke roop mein sirf word number dena hoga, one-hot encoding ke bina.

In [ ]:
vocab_size = len(vocab)

embedder = torch.nn.Embedding(num_embeddings = vocab_size, embedding_dim = 30)
model = torch.nn.Sequential(
    embedder,
    torch.nn.Linear(in_features = 30, out_features = vocab_size),
)

print(model)

## Training data taiyaar karna

Ab chaliye main function program karte hain, jo text se CBoW word pairs banayega. Yeh function humein window size specify karne dega aur input aur output words ke pairs ka ek set return karega. Dhyan rakhiye ki yeh function words par use ho sakta hai, aur vectors/tensors par bhi — jisse hum text ko encode kar sakte hain, uske `to_cbow` function mein pass hone se pehle.

In [ ]:
def to_cbow(sent,window_size=2):
    res = []
    for i,x in enumerate(sent):
        for j in range(max(0,i-window_size),min(i+window_size+1,len(sent))):
            if i!=j:
                res.append([sent[j],x])
    return res

print(to_cbow(['I','like','to','train','networks']))
print(to_cbow(encode('I like to train networks', vocab)))

Chaliye training dataset taiyaar karte hain. Hum saari news ko dekhenge, `to_cbow` ko call karenge taaki word pairs ki list mil sake, aur un pairs ko `X` aur `Y` mein add kar denge. Time bachane ke liye, hum sirf pehle 10k news items ko consider karenge — agar aapke paas zyada time hai aur behtar embeddings chahiye, toh aap is limit ko aasaani se hata sakte hain :)

In [ ]:
X = []
Y = []
for i, x in zip(range(10000), train_dataset):
    for w1, w2 in to_cbow(encode(x[1], vocab), window_size = 5):
        X.append(w1)
        Y.append(w2)

X = torch.tensor(X)
Y = torch.tensor(Y)

Hum us data ko ek dataset mein bhi convert karenge, aur dataloader banayenge:

In [ ]:
class SimpleIterableDataset(torch.utils.data.IterableDataset):
    def __init__(self, X, Y):
        super(SimpleIterableDataset).__init__()
        self.data = []
        for i in range(len(X)):
            self.data.append( (Y[i], X[i]) )
        random.shuffle(self.data)

    def __iter__(self):
        return iter(self.data)

Hum us data ko ek dataset mein convert karenge, aur dataloader banayenge:

In [ ]:
ds = SimpleIterableDataset(X, Y)
dl = torch.utils.data.DataLoader(ds, batch_size = 256)

Ab chaliye asli training karte hain. Hum `SGD` optimizer use karenge, ek kaafi high learning rate ke saath. Aap doosre optimizers, jaise `Adam`, ke saath bhi experiment kar sakte hain. Hum shuruaat mein 10 epochs ke liye train karenge — aur agar aap loss aur kam karna chahte hain, toh is cell ko dobara run kar sakte hain.

In [ ]:
def train_epoch(net, dataloader, lr = 0.01, optimizer = None, loss_fn = torch.nn.CrossEntropyLoss(), epochs = None, report_freq = 1):
    optimizer = optimizer or torch.optim.Adam(net.parameters(), lr = lr)
    loss_fn = loss_fn.to(device)
    net.train()

    for i in range(epochs):
        total_loss, j = 0, 0, 
        for labels, features in dataloader:
            optimizer.zero_grad()
            features, labels = features.to(device), labels.to(device)
            out = net(features)
            loss = loss_fn(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss
            j += 1
        if i % report_freq == 0:
            print(f"Epoch: {i+1}: loss={total_loss.item()/j}")

    return total_loss.item()/j

In [ ]:
train_epoch(net = model, dataloader = dl, optimizer = torch.optim.SGD(model.parameters(), lr = 0.1), loss_fn = torch.nn.CrossEntropyLoss(), epochs = 10)

## Word2Vec try karna

Word2Vec use karne ke liye, chaliye hamari vocabulary ke saare words ke liye vectors nikaalte hain:

In [ ]:
vectors = torch.stack([embedder(torch.tensor(vocab[s])) for s in vocab.itos], 0)

In [ ]:
paris_vec = embedder(torch.tensor(vocab['paris']))
print(paris_vec)

Word2Vec ka use karke synonyms dhoondhna kaafi interesting hai. Neeche wala function diye gaye input ke liye `n` sabse nazdeeki words return karega. Unhe dhoondhne ke liye hum $|w_i - v|$ ki value nikaalte hain, jahan $v$ hamare input word ka corresponding vector hai, aur $w_i$ vocabulary ke $i$-th word ki encoding hai. Iske baad hum array ko sort karte hain aur `argsort` use karke corresponding indices return karte hain, aur list ke pehle `n` elements lete hain, jo vocabulary mein sabse nazdeeki words ki position ko encode karte hain.

In [ ]:
def close_words(x, n = 5):
  vec = embedder(torch.tensor(vocab[x]))
  top5 = np.linalg.norm(vectors.detach().numpy() - vec.detach().numpy(), axis = 1).argsort()[:n]
  return [ vocab.itos[x] for x in top5 ]

close_words('microsoft')

In [ ]:
close_words('basketball')

In [ ]:
close_words('funds')

## Key Takeaways

CBoW jaisi smart techniques ka use karke, hum Word2Vec model ko train kar sakte hain. Aap skip-gram model ko bhi train karne ki koshish kar sakte hain, jise central word ke basis par neighboring word ko predict karne ke liye train kiya jaata hai, aur dekh sakte hain ki yeh kitna achha perform karta hai.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->